# 05d · 마팅게일 수렴과 폴리아 항아리 (Martingale Convergence and Pólya's Urn)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §5.5 (Convergence), §5.2 (Examples: Pólya urn, branching-process martingale) · Williams §11.1–11.5 (upcrossings, Doob's forward convergence theorem), §12.1–12.5 (martingales bounded in $L^2$), §13 (UI), §14.12 (Kakutani) · Lawler 2e §5.4–5.5 (UI, convergence), §2.4 (branching) · G&S 4e §12.3, §7.8 · Athreya & Ney (1972) Ch.1 (Kesten–Stigum, 인용만) |
| 선수 노트북 | 05c (월드 항등식·패턴 대기시간), 05a (마팅게일 정의·동물원: 폴리아·우도비·$Z_n/m^n$), 01g (분기과정의 소멸확률) |
| 예상 소요 | 90분 |
| 상태 | draft |

05a의 동물원에 있던 마팅게일들 — 폴리아 항아리의 빨간 비율, 골턴-왓슨의 $Z_n/m^n$, 우도비 $L_n$ — 은 모두 "시간이 지나면 어딘가에 **정착**한다". 이 노트북의 정리는 그 정착이 우연이 아니라는 것이다: **$L^1$ 유계 마팅게일은 거의 확실히 수렴한다.** 그런데 정착하는 값은 상수가 아니라 **확률변수**이고(폴리아: Beta$(a,b)$), 수렴한다고 해서 기대값이 보존되는 것도 아니다(우도비: $L_n\to0$인데 $E[L_n]=1$). 이 두 가지 "아닌 것"이 이 노트북에서 시뮬레이션으로 확인할 핵심이다. 대조군으로 수렴하지 **않는** 마팅게일(단순 랜덤워크)도 같이 돌린다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import betabinom, beta, norm, binom
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (galton_watson, extinction_probability, random_walk_paths,
                          simulate_chain, gamblers_ruin_matrix)
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("05d")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_DRAWS, N_GEN, N_LR, N_RW = 1000, 20, 200, 400   # 항아리 뽑기 수, GW 세대 수, 우도비 표본 수, SRW 걸음 수
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (05c) 월드 항등식(Wald's identity)을 진술하고 필요한 두 조건을 말하라.

<details><summary>정답</summary>

$E[S_T]=E[X_1]\,E[T]$. 조건: $T$가 정지시각이고 $E[T]<\infty$ (그리고 $E|X_1|<\infty$). 증명은 $S_n-n\mu$에 OST (c)를 적용한 것이다.

</details>

**Q2.** (05c) $E[T_{HHH}]=14$를 겹침 집합으로 설명하라.

<details><summary>정답</summary>

HHH의 접두부=접미부인 길이는 1, 2, 3 모두이므로 $2^1+2^2+2^3=14$. 시각 $T$에 살아 있는 도박꾼이 세 명(각각 $2,4,8$을 들고 있다)이고, 도박꾼 마팅게일에 OST를 적용하면 $E[T]=$ 그 합이다.

</details>

**Q3.** (05c) 기대 대기시간이 짧은 패턴이 Penney 게임에서 항상 먼저 나오는가?

<details><summary>정답</summary>

아니다. THH와 HHT는 둘 다 $E[T]=8$이지만 THH가 먼저 나올 확률은 $0.75$이고, 게임은 비추이적(non-transitive)이다. "평균 대기시간"과 "누가 먼저인가"는 다른 질문이다.

</details>

**Q4.** (05b) $M_{n\wedge T}$는 항상 마팅게일인데 왜 $E[M_T]=M_0$이 실패할 수 있는가?

<details><summary>정답</summary>

$n\to\infty$에서 $E[M_{n\wedge T}]\to E[M_T]$라는 극한-기대 교환이 실패할 수 있기 때문이다. 배가 전략에서는 질량 $2^{-n}$이 값 $-2^n$으로 도망가서 $E[M_{n\wedge T}]=1$이 극한 $E[M_T]$와 다르다. 이 노트북의 우도비 마팅게일은 같은 현상의 "정지시각 없는" 판이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **$L^1$ 유계**(bounded in $L^1$): $\sup_n E|M_n|<\infty$. **$L^2$ 유계**: $\sup_n E[M_n^2]<\infty$ (코시-슈바르츠로 $L^1$ 유계를 함의). **균등적분가능**(uniformly integrable, UI): $\lim_{K\to\infty}\sup_n E\big[|M_n|\,1_{\{|M_n|>K\}}\big]=0$. 포함 관계: $L^2$ 유계 ⇒ UI ⇒ $L^1$ 유계 (역은 모두 거짓). 유계($|M_n|\le c$)이면 당연히 UI.
- **상향 교차수**(upcrossings) $U_n[a,b]$: 시각 $n$까지 경로가 $a$ 아래에서 출발해 $b$ 위에 도달한 횟수 ($a<b$). 경로가 $a$ 이하를 찍으면 "장전", 그 뒤 $b$ 이상을 찍으면 교차 1회 완료. **둡 상향 교차 부등식**(Doob's upcrossing inequality): 슈퍼마팅게일 $M$에 대해
  $$(b-a)\,E[U_n[a,b]]\le E[(M_n-a)^-].$$
- **폴리아 항아리**(Pólya's urn): 초기 빨강 $a$개, 파랑 $b$개. 매번 한 개를 뽑아 **같은 색 한 개를 추가**해 되돌린다. $R_n$ = $n$번 뽑는 동안 추가된 빨강 수, $X_n=\dfrac{a+R_n}{a+b+n}$ = 빨간 비율. 05a Z5에서 $X_n$이 마팅게일임을 확인했다: $E[X_{n+1}\mid\mathcal F_n]=X_n\frac{a+R_n+1}{a+b+n+1}+(1-X_n)\frac{a+R_n}{a+b+n+1}=X_n$.
- **베타-이항 분포**(beta-binomial): $P(R_n=k)=\dbinom nk\dfrac{B(a+k,\,b+n-k)}{B(a,b)}=\dbinom nk\dfrac{a^{(k)}\,b^{(n-k)}}{(a+b)^{(n)}}$, 여기서 $x^{(k)}=x(x+1)\cdots(x+k-1)$은 상승 계승(rising factorial). 코드: `scipy.stats.betabinom(n, a, b)`.
- **골턴-왓슨 정규화**(normalised branching process): $W_n=Z_n/m^n$, $m=E[\text{자손 수}]$. 이 노트북의 자손 pmf는 $(p_0,p_1,p_2)=(0.2,0.3,0.5)$: $m=1.3$, $\sigma^2=0.61$, 소멸확률 $q=0.4$ (pgf 방정식 $0.2+0.3s+0.5s^2=s$의 작은 해; 01g).
- **우도비**(likelihood ratio): 관측 $X_i\sim g=N(0,1)$ i.i.d., 대립가설 $f=N(\mu,1)$, $\mu=0.5$. $L_n=\prod_{i\le n}\dfrac{f(X_i)}{g(X_i)}=\exp\!\big(\mu S_n-n\mu^2/2\big)$, $S_n=\sum_{i\le n}X_i$. 참 분포 $g$ 아래에서 $\log L_n\sim N(-n\mu^2/2,\;n\mu^2)$.

### 2.2 정리 1 — 마팅게일 수렴 정리 (Doob's forward convergence theorem; Williams §11.5, Durrett 3e §5.5, G&S 4e §12.3)

$M$이 $L^1$ 유계인 (서브/슈퍼)마팅게일이면 $M_n\to M_\infty$ a.s.이고 $E|M_\infty|<\infty$. 특히 **비음 슈퍼마팅게일**(따라서 비음 마팅게일)은 항상 a.s. 수렴한다.

가정이 왜 필요한가:
- **$L^1$ 유계**: 없으면 거짓이다. SRW $S_n$은 마팅게일이지만 $E|S_n|\sim\sqrt{2n/\pi}\to\infty$이고 수렴하지 않는다 — 모든 정수를 무한히 자주 방문한다(01x 재귀성). $L^1$ 유계는 상향 교차 부등식의 우변 $E[(M_n-a)^-]\le E|M_n|+|a|$를 통제하는 조건이다.
- **비음 슈퍼마팅게일이 자동으로 조건을 만족하는 이유**: $E|M_n|=E[M_n]\le E[M_0]<\infty$.
- **결론은 a.s. 수렴이지 $L^1$ 수렴이 아니다**: $E[M_\infty]=M_0$까지 얻으려면 UI가 있어야 한다(정리 5의 우도비가 반례).

증명 스케치 (상향 교차 부등식 자체는 Williams §11.2–11.3 / Durrett 3e §5.5 인용):

1. **베팅 전략으로 읽는 상향 교차 부등식.** "$M$이 $a$ 아래로 내려오면 1단위를 사고, $b$ 위로 올라가면 판다"는 전략 $H_n\in\{0,1\}$은 예측가능(시각 $n-1$까지의 정보로 결정)하다. 이 전략의 시각 $n$까지의 누적 이득 $Y_n=\sum_{k\le n}H_k(M_k-M_{k-1})$은 완료된 상향 교차마다 최소 $b-a$를 벌고, 마지막에 미완료 포지션이 있으면 기껏해야 $(M_n-a)^-$를 잃는다:
   $$Y_n\ \ge\ (b-a)\,U_n[a,b]-(M_n-a)^-.$$
   슈퍼마팅게일에 유계 비음 예측가능 전략으로 베팅한 이득은 다시 슈퍼마팅게일이므로 $E[Y_n]\le0$ (05b 정리 1의 논법). 기대를 취해 정리하면 부등식이다.
2. **$L^1$ 유계 ⇒ 교차 횟수 유한.** $(b-a)E[U_n[a,b]]\le E|M_n|+|a|\le\sup_nE|M_n|+|a|$. 단조수렴으로 $E[U_\infty[a,b]]<\infty$, 따라서 $U_\infty[a,b]<\infty$ a.s.
3. **진동 불가.** $\liminf M_n<a<b<\limsup M_n$인 경로는 $[a,b]$를 무한히 자주 상향 교차하므로 그런 경로의 확률은 0. 유리수 쌍 $a<b$는 가산개이므로 합집합의 확률도 0 ⇒ $\liminf M_n=\limsup M_n$ a.s. 즉 $M_n\to M_\infty\in[-\infty,\infty]$ a.s.
4. **극한이 유한.** 파투(Fatou): $E|M_\infty|\le\liminf E|M_n|\le\sup_nE|M_n|<\infty$ ⇒ $|M_\infty|<\infty$ a.s. $\square$

4.2에서 폴리아 경로들의 $U_{1000}[0.4,0.6]$을 직접 세어 (1)의 부등식을 숫자로 본다.

### 2.3 정리 2 — $L^2$ 유계 마팅게일의 수렴 (Williams §12.1, Durrett 3e §5.5)

$\sup_nE[M_n^2]<\infty$이면 $M_n\to M_\infty$ a.s.**이고 $L^2$에서도** 수렴하며
$$E[M_\infty]=M_0,\qquad \mathrm{Var}(M_\infty)=\sum_{k\ge1}E\big[(M_k-M_{k-1})^2\big].$$

가정이 왜 필요한가:
- **직교 증분** $E[(M_j-M_{j-1})(M_k-M_{k-1})]=0$ ($j<k$)이 핵심이고, 이것은 마팅게일 성질 자체에서 나온다(아래 증명 첫 줄).
- **$L^2$ 유계** ⇒ 증분 제곱합 수렴 ⇒ $L^2$ 코시 열 ⇒ $L^2$ 극한. $L^2$ 수렴은 $L^1$ 수렴을 함의하므로 기대값이 보존된다. 정리 1과 달리 "기대값 보존"이 공짜로 따라오는 것은 $L^2$ 유계가 UI를 함의하기 때문이다.

<details><summary>증명</summary>

$\Delta_k=M_k-M_{k-1}$이라 하자. $j<k$이면 $\Delta_j$는 $\mathcal F_{k-1}$-가측이므로 (아는 것 꺼내기, 탑 성질)

$$E[\Delta_j\Delta_k]=E\big[\Delta_j\,E[\Delta_k\mid\mathcal F_{k-1}]\big]=E[\Delta_j\cdot0]=0.$$

같은 논법으로 $E[M_0\Delta_k]=0$. 따라서 $M_n=M_0+\sum_{k\le n}\Delta_k$를 제곱해 기대를 취하면 교차항이 모두 사라져 **피타고라스 항등식**

$$E[M_n^2]=E[M_0^2]+\sum_{k=1}^nE[\Delta_k^2]$$

을 얻는다. $\sup_nE[M_n^2]<\infty$이면 $\sum_{k\ge1}E[\Delta_k^2]<\infty$. 그러면 $m<n$에 대해 $E[(M_n-M_m)^2]=\sum_{m<k\le n}E[\Delta_k^2]\to0$ ($m,n\to\infty$)이므로 $(M_n)$은 $L^2$에서 코시 열이고, $L^2$가 완비이므로 $L^2$ 극한 $M_\infty$가 존재한다. 한편 $L^2$ 유계 ⇒ $L^1$ 유계이므로 정리 1로 a.s. 극한도 존재하고, $L^2$ 수렴하는 열의 부분열은 a.s. 수렴하므로 두 극한은 a.s. 같다.

기대값: $|E[M_n]-E[M_\infty]|\le E|M_n-M_\infty|\le\|M_n-M_\infty\|_2\to0$이므로 $E[M_\infty]=\lim E[M_n]=M_0$. 분산: $E[M_\infty^2]=\lim E[M_n^2]=E[M_0^2]+\sum_kE[\Delta_k^2]$이고 $M_0$이 상수이면 $\mathrm{Var}(M_\infty)=\sum_kE[\Delta_k^2]$. $\square$

</details>

### 2.4 정리 3 — 폴리아 항아리 정리 (Durrett 3e §5.2, Williams §11.5 예제)

(i) $X_n$은 $[0,1]$-값 마팅게일이므로 $X_n\to X_\infty$ a.s.이고 $E[X_\infty]=a/(a+b)$.
(ii) $R_n\sim$ 베타-이항$(n,a,b)$: $P(R_n=k)=\dbinom nk\dfrac{a^{(k)}b^{(n-k)}}{(a+b)^{(n)}}$.
(iii) $X_\infty\sim\mathrm{Beta}(a,b)$. 예: $(a,b)=(1,1)$이면 균등분포, $(2,1)$이면 cdf $x^2$, $(5,5)$이면 $1/2$ 근처에 모인 종 모양.

가정이 왜 필요한가:
- **'같은 색 추가' 규칙**이 **교환가능성**(exchangeability: 어느 순서로 뽑히든 같은 확률)을 만든다. 반대 색을 추가하면 마팅게일도 교환가능성도 깨지고 극한이 상수 $1/2$가 된다(E3).
- **유계성** $0\le X_n\le1$ ⇒ UI ⇒ 기대값 보존 $E[X_\infty]=E[X_0]=a/(a+b)$. (정리 1만으로는 수렴까지만 나온다.)
- 극한 식별에는 "$[0,1]$ 위의 분포는 모멘트열로 결정된다"는 사실(모멘트 문제, 유계 지지집합에서는 항상 참)을 쓴다.

<details><summary>증명</summary>

**(i)** 마팅게일이고 $|X_n|\le1$이므로 $L^1$ 유계 ⇒ 정리 1로 a.s. 수렴. 유계수렴정리로 $E[X_\infty]=\lim E[X_n]=X_0=a/(a+b)$.

**(ii)** 처음 $k$번이 빨강, 그 다음 $n-k$번이 파랑일 확률은 뽑을 때마다 분모가 1씩 커지고 뽑힌 색의 분자가 1씩 커지므로

$$\frac{a}{a+b}\cdot\frac{a+1}{a+b+1}\cdots\frac{a+k-1}{a+b+k-1}\cdot\frac{b}{a+b+k}\cdots\frac{b+n-k-1}{a+b+n-1}=\frac{a^{(k)}\,b^{(n-k)}}{(a+b)^{(n)}}.$$

빨강 $k$개·파랑 $n-k$개를 **다른 순서**로 뽑아도 분모는 항상 $(a+b)(a+b+1)\cdots(a+b+n-1)$이고, 분자는 빨강이 뽑힐 때 $a,a+1,\dots,a+k-1$, 파랑이 뽑힐 때 $b,\dots,b+n-k-1$이 순서만 바뀌어 나타나므로 곱은 같다. 즉 뽑힌 색의 열은 교환가능하고, 빨강이 $k$개인 열은 $\binom nk$개 ⇒ (ii). ($B(a+k,b+n-k)/B(a,b)$ 형태는 $\Gamma(x+k)/\Gamma(x)=x^{(k)}$로 같은 식.)

**(iii)** $j$차 하강 계승 모멘트를 교환가능성으로 계산한다. $R_n=\sum_{i\le n}I_i$ ($I_i$ = $i$번째가 빨강)이면 $R_n(R_n-1)\cdots(R_n-j+1)$은 서로 다른 순서쌍 $(i_1,\dots,i_j)$ 중 모두 빨강인 것의 개수이고, 교환가능성으로 특정 $j$개 위치가 모두 빨강일 확률은 "처음 $j$번이 모두 빨강"일 확률 $a^{(j)}/(a+b)^{(j)}$와 같다:

$$E\big[R_n(R_n-1)\cdots(R_n-j+1)\big]=n(n-1)\cdots(n-j+1)\,\frac{a^{(j)}}{(a+b)^{(j)}}.$$

양변을 $(a+b+n)^j$로 나누고 $n\to\infty$: 좌변은 $E[X_n^j]+O(1/n)$이고 우변의 $n(n-1)\cdots(n-j+1)/(a+b+n)^j\to1$이므로

$$E[X_n^j]\ \to\ \frac{a^{(j)}}{(a+b)^{(j)}}=\frac{B(a+j,b)}{B(a,b)}=E[Y^j],\qquad Y\sim\mathrm{Beta}(a,b).$$

$[0,1]$ 위의 분포는 모멘트열로 유일하게 결정되고 모멘트 수렴은 분포 수렴을 주므로 $X_n\Rightarrow\mathrm{Beta}(a,b)$. 그런데 (i)에서 $X_n\to X_\infty$ a.s. ⇒ 분포 수렴이기도 하므로 극한 분포는 같아야 한다: $X_\infty\sim\mathrm{Beta}(a,b)$. $\square$

유한 $n$에서 $X_n$의 분포는 Beta가 **아니라** 베타-이항을 $1/(a+b+n)$로 스케일한 것이다. 5절의 정확값은 모두 `betabinom`으로 계산한다.

</details>

### 2.5 정리 4 — 골턴-왓슨 마팅게일의 수렴과 Kesten–Stigum (Durrett 3e §5.2, Lawler 2e §2.4, Athreya & Ney Ch.1)

$m>1$, $\sigma^2<\infty$이면 $W_n=Z_n/m^n$은 $L^2$ 유계 마팅게일이고
$$\mathrm{Var}(W_n)=\frac{\sigma^2(1-m^{-n})}{m(m-1)},\qquad W_n\to W\ \text{a.s. \& } L^2,\qquad E[W]=1,\quad \mathrm{Var}(W)=\frac{\sigma^2}{m(m-1)},\quad P(W=0)=q.$$
(Kesten–Stigum: $\sigma^2<\infty$ 없이도 $E[W]=1\iff E[X\log^+X]<\infty$, $X$ = 자손 수. 여기서는 인용만.)

가정이 왜 필요한가:
- **$m>1$**: 아임계·임계($m\le1$)에서는 $Z_n\to0$ a.s.이므로 $W=0$ — 여전히 수렴은 하지만 자명하다(비음 마팅게일이니 정리 1은 여전히 적용된다).
- **$\sigma^2<\infty$**: $L^2$ 논증에 필요. 유계 자손 분포(여기서는 $\le2$)는 자동 만족. Kesten–Stigum은 이 조건을 $E[X\log X]<\infty$로 약화한다.
- **$P(W=0)=q$**: $\{W=0\}\supseteq\{\text{소멸}\}$이라 $P(W=0)\ge q$인데, $E[W]=1$이면 $P(W=0)<1$이고 $P(W=0)$이 pgf의 고정점 방정식을 만족하므로 작은 해 $q$와 같다.

증명 스케치 (분산 재귀는 완전 증명, 나머지는 정리 2 적용):

1. **분산 재귀.** $Z_{n+1}=\sum_{i\le Z_n}\xi_i$ (i.i.d. 자손 수)에 조건부 분산 공식 $\mathrm{Var}(Z_{n+1})=E[\mathrm{Var}(Z_{n+1}\mid Z_n)]+\mathrm{Var}(E[Z_{n+1}\mid Z_n])=E[Z_n\sigma^2]+\mathrm{Var}(mZ_n)=\sigma^2m^n+m^2\mathrm{Var}(Z_n)$. $\mathrm{Var}(Z_0)=0$에서 풀면 $\mathrm{Var}(Z_n)=\sigma^2m^{n-1}(1+m+\cdots+m^{n-1})=\sigma^2m^{n-1}\dfrac{m^n-1}{m-1}$. $m^{2n}$으로 나누면 $\mathrm{Var}(W_n)=\dfrac{\sigma^2(1-m^{-n})}{m(m-1)}\le\dfrac{\sigma^2}{m(m-1)}$ — $L^2$ 유계.
2. **정리 2 적용.** $W_n\to W$ a.s.와 $L^2$, $E[W]=\lim E[W_n]=1$, $\mathrm{Var}(W)=\lim\mathrm{Var}(W_n)=\sigma^2/(m(m-1))$.
3. **$P(W=0)=q$.** 1세대 개체 $Z_1=k$명이 각각 독립인 부분 과정을 낳으므로 $W=m^{-1}\sum_{j\le k}W^{(j)}$ ($W^{(j)}$는 $W$의 i.i.d. 복제). 따라서 $P(W=0)=\sum_kp_k\,P(W=0)^k=\varphi(P(W=0))$: $P(W=0)$은 pgf $\varphi$의 고정점이고, $m>1$이면 고정점은 $q$와 $1$뿐이다. $E[W]=1$이므로 $P(W=0)\ne1$ ⇒ $P(W=0)=q$. $\square$

수치 (5절에서 코드로 재계산): $(0.2,0.3,0.5)$에서 $\mathrm{Var}(W_{20})=1.55587$, $\mathrm{Var}(W)=1.56410$, $P(Z_{20}=0)=\varphi^{(20)}(0)=0.399850$ (극한 $q=0.4$).

### 2.6 정리 5 — 우도비 마팅게일의 소멸 (Kakutani; Williams §14.12, Durrett 3e §5.2)

$X_i\sim g$ i.i.d., $f\ne g$이면 $L_n=\prod_{i\le n}f(X_i)/g(X_i)$에 대해 $L_n\to0$ a.s.이지만 $E_g[L_n]=1$ (모든 $n$). 따라서 $E[L_\infty]=0\ne1$: **$L_n$은 UI가 아니다.** 가우스 경우 $\frac1n\log L_n\to-\mu^2/2=-0.125$.

가정이 왜 필요한가:
- **$f\ne g$**: 쿨백-라이블러 발산 $KL(g\|f)=E_g[\log(g/f)]>0$이어야 $\log L_n$이 $-\infty$로 간다. $KL\ge0$은 젠센 부등식이고 등호는 $f=g$ a.e.일 때뿐이다.
- **$E_g|\log(f/g)|<\infty$**: 강대수법칙(SLLN)을 쓰기 위한 조건. 가우스 경우 $\log(f/g)(x)=\mu x-\mu^2/2$이므로 명백.

<details><summary>증명</summary>

$L_n$은 비음 마팅게일(05a Z6: $E_g[f(X)/g(X)]=\int f=1$)이므로 정리 1로 a.s. 수렴한다. 극한을 식별하기 위해 로그를 취한다:

$$\frac1n\log L_n=\frac1n\sum_{i\le n}\log\frac{f(X_i)}{g(X_i)}\ \xrightarrow{\text{SLLN}}\ E_g\Big[\log\frac fg\Big]=-KL(g\|f)<0.$$

따라서 $\log L_n\to-\infty$ a.s., 즉 $L_n\to0$ a.s. 가우스 경우: $\log\frac fg(x)=\mu x-\mu^2/2$이고 $E_g[X]=0$이므로 극한은 $-\mu^2/2$.

한편 $E[L_n]=L_0=1$은 마팅게일 성질이다. 만약 $(L_n)$이 UI였다면 a.s. 수렴 + UI ⇒ $L^1$ 수렴이므로 $E[L_\infty]=\lim E[L_n]=1$이어야 하는데 $L_\infty=0$이므로 모순. 따라서 UI가 아니다. $\square$

숫자로 보면: $\log L_n\sim N(-n\mu^2/2,\,n\mu^2)$이므로 $n=200$, $\mu=0.5$에서 $P(L_{200}<0.01)=\Phi\big((\ln0.01+25)/\sqrt{50}\big)=\Phi(2.884)=0.998038$. 즉 99.8%의 경로에서 $L_{200}<0.01$인데도 평균은 정확히 1이다 — 질량이 $e^{25}$ 규모의 희귀 사건에 있다. 이것은 05b 배가 전략의 "질량 $2^{-n}$이 $-2^n$으로"의 연속판이다.

</details>

### 2.7 직관

- **수렴 정리의 그림.** $L^1$ 유계 마팅게일은 어떤 구간 $[a,b]$도 무한히 자주 건널 수 없다. 건널 때마다 "싸게 사서 비싸게 파는" 전략이 공정한 게임에서 확실히 돈을 벌게 되는데, 공정한 게임에서는 어떤 전략도 기대 이득이 0 이하이기 때문이다. 진동이 불가능하면 남는 것은 수렴뿐이다.
- **폴리아 항아리는 '부익부' 모형.** 초기 몇 번의 뽑기가 운명을 정하고, 그 뒤에는 (조건부로) 큰 수의 법칙처럼 굳어진다. 그래서 극한이 상수가 아니라 Beta 확률변수다: 같은 항아리를 다시 돌리면 다른 값에 정착한다.
- **골턴-왓슨.** 초기 세대의 운이 $W$를 정하고 그 뒤는 $m^n$ 지수 성장이 거의 결정론적으로 이어진다. $W$는 "초기 운의 총량"이고, $W=0$은 소멸이다.
- **우도비는 통계학의 마팅게일.** 참 분포 아래에서 틀린 가설의 우도비는 0으로 간다(검정력이 1로 간다는 뜻). 그런데 기대는 1로 고정 — 질량이 극히 드문 큰 값으로 도망가는 배가 전략의 연속판이다. **MC에 주는 교훈**: $E[L_{200}]=1$을 표본 평균으로 추정하면 분산이 $e^{50}-1$이라 거의 항상 1보다 훨씬 작은 값이 나온다. 불편추정량이라도 분산이 폭발하면 쓸모가 없다.

✍️ 내 말로: "수렴한다"와 "기대값이 보존된다"가 다른 말인 이유를 우도비 예로 세 문장으로 써 보세요.

### 2.8 함정

- **a.s. 수렴 ≠ 기대값 보존.** $L_n\to0$인데 $E[L_n]=1$. 더 단순한 예: $Y_i\in\{0,2\}$ 등확률의 곱 $M_n=\prod Y_i$도 $M_n\to0$ a.s., $E[M_n]=1$ (E1c). 보존에는 UI가 필요하다.
- **폴리아 항아리의 극한은 확률변수다.** "많이 뽑으면 비율이 $a/(a+b)$로 간다"는 오해 — 평균만 그렇다.
- **$W$의 분포는 0에 원자를 갖는다.** $P(W=0)=q=0.4$이고 $\{W>0\}$ 위에서는 연속. 히스토그램을 그릴 때 0을 분리해 보고해야 한다(그림 4).
- **$E[L_{200}]=1$의 MC 추정은 무의미하다** — 분산 $e^{50}-1$. 그래서 assert하지 않는다. 대신 $E[L_5]=1$ (분산 $e^{1.25}-1=2.49$)을 assert하고 큰 $n$은 그림으로만 본다.
- **SRW $S_n$은 마팅게일이지만 수렴하지 않는다.** 수렴 정리는 "모든 마팅게일"이 아니라 "$L^1$ 유계 마팅게일"에 대한 것이다.
- **유한 $n$에서 $X_n$의 분포는 Beta가 아니다.** 베타-이항의 스케일이고, Beta cdf와는 $O(1/n)$ 차이가 난다: $(1,1)$에서 $P(X_{1000}\le1/4)=250/1001=0.24975$ vs 극한 $1/4$. assert의 정확값은 `betabinom`으로 계산해야 한다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

돌리기 전에 숫자를 적어 보세요. 정리 3(베타-이항), 정리 4(분산 공식·pgf 반복), 정리 5(로그 정규)로 모두 손 계산이 가능합니다. 맞고 틀림보다 "어느 정리를 썼는가"가 중요합니다.

In [ ]:
predictions = {
    # (a,b)=(2,1) 항아리에서 E[X_1000]은? (마팅게일이므로 X_0과 같다)
    "polya_21_mean": None,
    # (2,1)에서 P(X_1000 <= 1/2)는? (극한 Beta(2,1)의 cdf는 x^2; 유한 n은 betabinom)
    "polya_21_p_le_half": None,
    # (1,1)에서 P(X_1000 <= 1/4)는? (R_1000은 {0,...,1000} 위 균등)
    "polya_11_p_le_quarter": None,
    # 자손 pmf (0.2,0.3,0.5)에서 P(Z_20 = 0)은? (q = 0.4; 20세대면 아직 q에 못 미친다)
    "gw_p_extinct_20": None,
    # Var(W_20)은? (W_20 = Z_20 / 1.3^20, 정리 4의 분산 공식)
    "gw_var_W20": None,
    # mu=0.5 가우스 우도비에서 P_g(L_200 < 0.01)은? (log L_200 ~ N(-25, 50); 그런데 E[L_200] = 1)
    "lr_p_small": None,
}

## 4. Simulate — 시뮬레이션

이 노트북에서만 쓰는 도우미 세 개: 폴리아 항아리 경로 생성기(05a와 같은 벡터화 루프), 상향 교차수 세기(경로별 상태기계를 시간축 루프 하나로), $P(Z_n=0)=\varphi^{(n)}(0)$의 pgf 반복.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def polya_urn(a, b, n_draws, rng, n_paths):
    """폴리아 항아리: 빨강 a·파랑 b로 시작, 뽑은 색 1개 추가. 빨간 비율 X_n, shape (n_paths, n_draws+1)."""
    red, total = np.full(n_paths, a, dtype=float), float(a + b)
    X = np.empty((n_paths, n_draws + 1)); X[:, 0] = red / total
    for k in range(n_draws):
        red += rng.random(n_paths) < red / total     # 빨강을 뽑으면 빨강 +1
        total += 1.0
        X[:, k + 1] = red / total
    return X

def count_upcrossings(X, a, b):
    """각 경로의 [a,b] 상향 교차 완료 횟수 U_n[a,b] (a 이하를 찍은 뒤 b 이상 도달 = 1회)."""
    armed = np.zeros(X.shape[0], dtype=bool)         # a 이하를 찍어 '장전'된 상태
    U = np.zeros(X.shape[0], dtype=int)
    for k in range(X.shape[1]):
        x = X[:, k]
        done = armed & (x >= b)
        U += done
        armed = (armed & ~done) | (x <= a)
    return U

def gw_extinction_by_generation(pmf, n):
    """P(Z_n = 0) = phi^(n)(0): pgf를 0에서 n번 반복."""
    pmf = np.asarray(pmf, dtype=float); ks = np.arange(pmf.size)
    q = 0.0
    for _ in range(n):
        q = float(np.sum(pmf * q**ks))
    return q

### 4.1 폴리아 항아리 — 경로는 정착하지만 정착값은 제각각 (Fig 1, 2)

세 항아리 $(a,b)\in\{(1,1),(2,1),(5,5)\}$를 각각 $N_{\text{PATHS}}$개, 1000번 뽑기까지 돌린다. 먼저 $(1,1)$ 경로 15개를 log-x 축으로 본다: 초기 열 번 정도에서 크게 흔들리다가 $n\ge100$부터 각자 다른 값에 굳는다.

In [ ]:
URNS = [(1, 1), (2, 1), (5, 5)]
X = {ab: polya_urn(*ab, N_DRAWS, rng, N_PATHS) for ab in URNS}
n_grid = np.arange(N_DRAWS + 1)

fig, ax = plt.subplots()
plot_paths(n_grid[1:], X[(1, 1)][:15, 1:], ax=ax, n_show=15, alpha=0.7)
ax.set_xscale("log"); ax.set_ylim(0, 1)
ax.set_title("Fig 1. Polya urn (1,1): 15 paths of the red fraction X_n (log-x axis)")
ax.set_xlabel("n (draws)"); ax.set_ylabel("X_n");

정리 3 (iii)의 예측: $X_{1000}$의 히스토그램은 Beta$(a,b)$ 밀도와 거의 일치해야 한다 — $(1,1)$은 평평, $(2,1)$은 직선 $2x$, $(5,5)$는 종 모양.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
xs = np.linspace(0.001, 0.999, 400)
for ax, (a, b) in zip(axes, URNS):
    plot_hist_vs_pdf(X[(a, b)][:, -1], xs, beta(a, b).pdf(xs), ax=ax, bins=40, label_pdf=f"Beta({a},{b}) pdf")
    ax.set_title(f"Fig 2. X_1000 histogram, urn ({a},{b})"); ax.set_xlabel("X_1000"); ax.legend()
fig.tight_layout()

측정: 평균(마팅게일 ⇒ $a/(a+b)$), $(2,1)$의 분산, 그리고 두 꼬리확률. 분산의 SE는 제곱편차 $(x-\bar x)^2$의 표본 SE로 근사한다.

In [ ]:
X11, X21, X55 = X[(1, 1)][:, -1], X[(2, 1)][:, -1], X[(5, 5)][:, -1]
est_polya = {
    "mean21": mc_estimate(X21),
    "var21": mc_estimate((X21 - X21.mean()) ** 2),
    "p21": mc_proportion(X21 <= 0.5),
    "p11": mc_proportion(X11 <= 0.25),
    "mean55": mc_estimate(X55),
}
for k, v in est_polya.items():
    print(f"{k:>7}: {v}")

### 4.2 상향 교차수 — 정리 1의 부등식을 숫자로 (Fig 3)

$(1,1)$ 경로들에서 구간 $[0.4,0.6]$의 상향 교차수 $U_{1000}[0.4,0.6]$을 센다. 정리 1의 부등식은 $0.2\,E[U]\le E[(X_{1000}-0.4)^-]$이고, 우변은 (거칠게) $\le0.4$이므로 $E[U]\le2$; 정확한 우변은 `betabinom`으로 $E[(0.4-X_{1000})^+]$를 계산해 얻는다. 대부분의 경로가 0회 또는 1회로 끝나야 한다.

In [ ]:
U = count_upcrossings(X[(1, 1)], 0.4, 0.6)
est_U = mc_estimate(U)
d11 = betabinom(N_DRAWS, 1, 1)
x_k = (1 + np.arange(N_DRAWS + 1)) / (2 + N_DRAWS)                      # R=k 일 때의 X_1000
neg_part = float(np.sum(d11.pmf(np.arange(N_DRAWS + 1)) * np.clip(0.4 - x_k, 0, None)))
bound_sharp, bound_crude = neg_part / 0.2, 0.4 / 0.2

vals, counts = np.unique(U, return_counts=True)
fig, ax = plt.subplots()
ax.bar(vals, counts / len(U), width=0.6)
ax.set_title("Fig 3. Number of upcrossings of [0.4, 0.6] by n=1000 (Polya (1,1))")
ax.set_xlabel("U_1000[0.4, 0.6]"); ax.set_ylabel("fraction of paths"); ax.set_xticks(vals)
ax.text(0.98, 0.9, f"E[U] = {est_U.mean:.3f}\nDoob bound: E[U] <= {bound_sharp:.3f} (exact), <= {bound_crude:.0f} (crude)",
        ha="right", va="top", transform=ax.transAxes)
print(f"E[U_1000] = {est_U},  bounds: {bound_sharp:.4f} (exact rhs), {bound_crude:.1f} (crude rhs)")

부등식은 넉넉히 성립한다(추정 $\approx0.19$ vs 정확 우변 $\approx0.40$). 중요한 것은 부등식의 상수가 아니라 "$n$을 늘려도 $E[U_n]$의 상한이 $n$에 무관하게 유계"라는 사실 — 이것이 무한 진동을 막는다.

### 4.3 골턴-왓슨 $W_n=Z_n/m^n$ — 0의 원자와 연속 부분 (Fig 4)

자손 pmf $(0.2,0.3,0.5)$, 20세대. 왼쪽: 계보 20개의 $W_n$ — 소멸하는 계보는 0에 붙고, 살아남는 계보는 초기 몇 세대에서 정해진 값 근처에 굳는다. 오른쪽: $W_{20}>0$인 경로만의 히스토그램(0의 원자는 텍스트로).

In [ ]:
PMF = np.array([0.2, 0.3, 0.5])
m = float(PMF @ np.arange(3)); sigma2 = float(PMF @ np.arange(3) ** 2 - m ** 2)
Z = galton_watson(PMF, N_GEN, rng, N_PATHS)
W = Z / m ** np.arange(N_GEN + 1)
W20 = W[:, -1]
p0_hat = mc_proportion(Z[:, -1] == 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
plot_paths(np.arange(N_GEN + 1), W, ax=ax1, n_show=20, alpha=0.6)
ax1.set_title(f"Fig 4a. W_n = Z_n / m^n for 20 lineages (m={m:.1f})"); ax1.set_xlabel("generation n"); ax1.set_ylabel("W_n")
ax2.hist(W20[W20 > 0], bins=40, density=True, alpha=0.6, label="W_20 given W_20 > 0")
ax2.set_title(f"Fig 4b. Histogram of W_20 given W_20 > 0; atom at 0 has mass {p0_hat.mean:.4f}")
ax2.set_xlabel("W_20"); ax2.legend()
print(f"m={m}, sigma^2={sigma2:.2f}, m^20={m**N_GEN:.2f}, max W_20={W20.max():.2f}")

In [ ]:
est_gw = {
    "E[W_20]": mc_estimate(W20),
    "P(Z_20=0)": p0_hat,
    "Var(W_20)": mc_estimate((W20 - W20.mean()) ** 2),
}
q20 = gw_extinction_by_generation(PMF, N_GEN)         # phi^(20)(0)
q_inf = extinction_probability(PMF)                     # q = 0.4
for k, v in est_gw.items():
    print(f"{k:>10}: {v}")
print(f"exact P(Z_20=0) = {q20:.6f},  q = {q_inf:.6f},  Var(W_20) formula = {sigma2*(1-m**-N_GEN)/(m*(m-1)):.6f}")

### 4.4 우도비 — 0으로 수렴하지만 평균은 1 (Fig 5)

$X_i\sim N(0,1)$, $\mu=0.5$: $\log L_n=\mu S_n-n\mu^2/2$. 왼쪽은 $\log L_n$ 경로 20개와 SLLN의 직선 $-\mu^2n/2$. 오른쪽은 각 $n$에서 경로들의 표본평균 $\bar L_n$ (log-y): 참값은 모든 $n$에서 1이지만, 표본평균은 $n$이 커지면서 1 아래로 무너진다 — 추정량은 불편이지만 분산 $(e^{n\mu^2}-1)/N_{\text{PATHS}}$이 폭발하기 때문이다.

In [ ]:
MU = 0.5
G = rng.standard_normal((N_PATHS, N_LR))
logL = MU * np.cumsum(G, axis=1) - MU ** 2 / 2 * np.arange(1, N_LR + 1)   # log L_n, n=1..200
n_lr = np.arange(1, N_LR + 1)
Lbar = np.exp(logL).mean(axis=0)                                           # 각 n의 표본평균 (참값 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
plot_paths(n_lr, logL[:20], ax=ax1, n_show=20, alpha=0.5)
ax1.plot(n_lr, -MU ** 2 / 2 * n_lr, "k--", lw=2, label="-mu^2 n / 2 (SLLN)")
ax1.set_title("Fig 5a. log L_n for 20 paths vs n (mu = 0.5)"); ax1.set_xlabel("n"); ax1.set_ylabel("log L_n"); ax1.legend()
ax2.semilogy(n_lr, Lbar, label=f"sample mean of L_n over {N_PATHS} paths")
ax2.axhline(1.0, color="k", ls="--", label="true E[L_n] = 1")
ax2.set_title("Fig 5b. Sample mean of L_n vs n (log y)"); ax2.set_xlabel("n"); ax2.legend()
print(f"sample mean of L_200 = {Lbar[-1]:.4g}  (true value 1);  fraction of paths with L_200 < 0.01 = {(logL[:, -1] < np.log(0.01)).mean():.4f}")

측정은 두 가지만: $E[L_5]=1$ (분산 $e^{1.25}-1\approx2.49$라 assert 가능)과 $P(L_{200}<0.01)$. $E[L_{200}]$은 표에 넣지 않는다 — 표본평균이 무엇을 주든 그 값은 정보가 아니다.

In [ ]:
est_lr = {
    "E[L_5]": mc_estimate(np.exp(logL[:, 4])),
    "P(L_200<0.01)": mc_proportion(logL[:, -1] < np.log(0.01)),
}
for k, v in est_lr.items():
    print(f"{k:>14}: {v}")

### 4.5 수렴하지 않는 마팅게일 vs 유계 마팅게일 (Fig 6)

대조군. 왼쪽: SRW의 $E|S_n|$ vs $n$ — $\sqrt{2n/\pi}$를 따라 발산한다($L^1$ 무계 ⇒ 정리 1 적용 불가, 실제로 수렴하지 않는다). 오른쪽: 같은 SRW를 $\{0,10\}$에서 멈춘 파산 사슬 $S_{n\wedge T}$ — $[0,10]$에 갇힌 유계 마팅게일이라 반드시 수렴하고, 극한은 $0$ 또는 $10$뿐이다.

In [ ]:
S = random_walk_paths(N_PATHS, N_RW, rng)
n_rw = np.arange(N_RW + 1)
absS_mean = np.abs(S).mean(axis=0)
est_S100 = mc_estimate(np.abs(S[:, 100]))
P_ruin = gamblers_ruin_matrix(10, 0.5)
R = simulate_chain(P_ruin, 3, N_RW, rng, n_paths=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))
ax1.plot(n_rw, absS_mean, label="MC E|S_n|")
ax1.plot(n_rw, np.sqrt(2 * n_rw / np.pi), "k--", label="sqrt(2n/pi)")
ax1.set_title("Fig 6a. E|S_n| vs n: SRW is not L^1 bounded"); ax1.set_xlabel("n"); ax1.set_ylabel("E|S_n|"); ax1.legend()
plot_paths(n_rw, R, ax=ax2, n_show=10, alpha=0.7)
ax2.set_title("Fig 6b. Bounded martingale S_{n AND T} (ruin chain, N=10, start 3): 10 paths")
ax2.set_xlabel("n"); ax2.set_ylabel("S_{n AND T}")
print(f"E|S_100| = {est_S100};  final states of the 10 ruin paths: {R[:, -1]}")

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산한다: 폴리아는 `betabinom(1000, a, b)`의 평균·분산·cdf, 골턴-왓슨은 $\varphi^{(20)}(0)$과 $\sigma^2(1-m^{-20})/(m(m-1))$, 우도비는 $\Phi\big((\ln0.01+n\mu^2/2)/\sqrt{n\mu^2}\big)$, SRW는 $E|S_{100}|=\sum_k\binom{100}k2^{-100}|2k-100|$.

In [ ]:
d21, d55 = betabinom(N_DRAWS, 2, 1), betabinom(N_DRAWS, 5, 5)
k100 = np.arange(101)
exact = {
    "mean21": (2 + d21.mean()) / (3 + N_DRAWS),                  # = 2/3 (마팅게일)
    "var21": d21.var() / (3 + N_DRAWS) ** 2,                     # 극한 1/18 보다 O(1/n) 작다
    "p21": d21.cdf(499),                                         # X_1000 <= 1/2  <=>  R_1000 <= 499
    "p11": d11.cdf(249),                                         # = 250/1001 (R_1000 균등)
    "mean55": (5 + d55.mean()) / (10 + N_DRAWS),                 # = 1/2
    "EW": 1.0,
    "p0": q20,
    "varW20": sigma2 * (1 - m ** -N_GEN) / (m * (m - 1)),
    "L5": 1.0,
    "p200": norm.cdf((np.log(0.01) + N_LR * MU ** 2 / 2) / np.sqrt(N_LR * MU ** 2)),
    "S100": float(np.sum(binom.pmf(k100, 100, 0.5) * np.abs(2 * k100 - 100))),
}
print({k: round(v, 6) for k, v in exact.items()})

In [ ]:
rows = [
    {"name": "E[X_1000], Polya(2,1)", "predicted": predictions["polya_21_mean"], "estimate": est_polya["mean21"], "exact": exact["mean21"]},
    {"name": "Var(X_1000), Polya(2,1)", "predicted": None, "estimate": est_polya["var21"], "exact": exact["var21"]},
    {"name": "P(X_1000 <= 1/2), Polya(2,1)", "predicted": predictions["polya_21_p_le_half"], "estimate": est_polya["p21"], "exact": exact["p21"]},
    {"name": "P(X_1000 <= 1/4), Polya(1,1)", "predicted": predictions["polya_11_p_le_quarter"], "estimate": est_polya["p11"], "exact": exact["p11"]},
    {"name": "E[X_1000], Polya(5,5)", "predicted": None, "estimate": est_polya["mean55"], "exact": exact["mean55"]},
    {"name": "E[W_20], GW", "predicted": None, "estimate": est_gw["E[W_20]"], "exact": exact["EW"]},
    {"name": "P(Z_20 = 0), GW", "predicted": predictions["gw_p_extinct_20"], "estimate": est_gw["P(Z_20=0)"], "exact": exact["p0"]},
    {"name": "Var(W_20), GW", "predicted": predictions["gw_var_W20"], "estimate": est_gw["Var(W_20)"], "exact": exact["varW20"]},
    {"name": "E_g[L_5], Gaussian LR", "predicted": None, "estimate": est_lr["E[L_5]"], "exact": exact["L5"]},
    {"name": "P_g(L_200 < 0.01)", "predicted": predictions["lr_p_small"], "estimate": est_lr["P(L_200<0.01)"], "exact": exact["p200"]},
    {"name": "E abs(S_100), SRW (not L^1 bounded)", "predicted": None, "estimate": est_S100, "exact": exact["S100"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **E[X_1000], Polya(2,1) / (5,5)**: 마팅게일 성질 자체. 어긋나면 `polya_urn`이 뽑은 색이 아닌 색을 추가하고 있다(E3의 항아리가 되어 평균이 1/2로 끌려간다).
- **Var(X_1000), Polya(2,1)**: 정확값은 betabinom 분산 $/1003^2$. 극한값 $1/18=0.05556$을 넣으면 $0.3\%$ 차이(약 0.3 SE)라 눈에 안 띈다 — 이 행은 유한-$n$ 보정의 존재를 확인하는 용도다.
- **P(X_1000 ≤ 1/2), (2,1) / P(X_1000 ≤ 1/4), (1,1)**: 정리 3 (ii)-(iii). 어긋나면 `<=`와 `<`의 경계 처리($R_{1000}\le499$)나 $X_n$의 분모 $a+b+n$이 틀렸다.
- **E[W_20], GW**: $W_n$의 마팅게일 성질. 어긋나면 $m^n$의 지수가 밀렸다($Z$의 열 인덱스 0이 0세대).
- **P(Z_20=0), GW**: pgf 반복 20회. $q=0.4$를 넣으면 $0.00015$ 차이로 표에서는 구분이 안 되지만 원리상 다르다.
- **Var(W_20), GW**: 정리 4의 분산 재귀. $\mathrm{Var}(W)=\sigma^2/(m(m-1))$ 극한값을 넣으면 $0.5\%$ 차이(FAST에서는 SE 안, 전체 실행에서는 약 0.4 SE). 어긋나면 $\sigma^2$를 $E[\xi^2]$로 잘못 계산했을 가능성.
- **E_g[L_5]**: 마팅게일 성질 $E[L_n]=1$. 어긋나면 $\log L_n$의 보정항 $-n\mu^2/2$가 빠졌거나 부호가 틀렸다. ($n=200$이었다면 어긋나는 것이 정상 — 분산 폭발.)
- **P_g(L_200 < 0.01)**: $\log L_{200}\sim N(-25,50)$. 어긋나면 `np.cumsum`의 축이 틀려 $S_n$이 경로를 가로질러 합해졌다.
- **E|S_100|**: 이항 합. 어긋나면 `random_walk_paths`의 걸음이 $\pm1$이 아니다. 이 행은 "수렴하지 않는다"의 정량 근거: $n=400$이면 $\approx16$으로 두 배가 된다.

In [ ]:
assert_close(est_polya["mean21"], exact["mean21"], k=4, name="Polya(2,1) E[X_1000]")
assert_close(est_polya["var21"], exact["var21"], k=4, name="Polya(2,1) Var(X_1000) (betabinom exact)")
assert_close(est_polya["p21"], exact["p21"], k=4, name="Polya(2,1) P(X_1000<=1/2)")
assert_close(est_polya["p11"], exact["p11"], k=4, name="Polya(1,1) P(X_1000<=1/4)")
assert_close(est_polya["mean55"], exact["mean55"], k=4, name="Polya(5,5) E[X_1000]")
assert_close(est_gw["E[W_20]"], exact["EW"], k=4, name="GW E[W_20]")
assert_close(est_gw["P(Z_20=0)"], exact["p0"], k=4, name="GW P(Z_20=0)")
assert_close(est_gw["Var(W_20)"], exact["varW20"], k=4, name="GW Var(W_20)")
assert_close(est_lr["E[L_5]"], exact["L5"], k=4, name="E_g[L_5]")
assert_close(est_lr["P(L_200<0.01)"], exact["p200"], k=4, name="P_g(L_200<0.01)")
assert_close(est_S100, exact["S100"], k=4, name="E abs(S_100)")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

(a) $(a,b)=(1,2)$ 항아리의 극한 분포, $P(X_\infty\le0.3)$, $E[X_\infty]$, $\mathrm{Var}(X_\infty)$를 구하라.
(b) 자손 pmf $(0.2,0.3,0.5)$에서 $\mathrm{Var}(W)$와 $P(W=0)$을 구하라.
(c) $Y_i\in\{0,2\}$ 등확률 i.i.d.의 곱 $M_n=\prod_{i\le n}Y_i$에 대해 $P(M_n>0)$, $E[M_n]$, a.s. 극한을 구하라. 이 예가 정리 1의 어떤 부분을 보여 주는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 정리 3: $X_\infty\sim\mathrm{Beta}(1,2)$, 밀도 $2(1-x)$, cdf $1-(1-x)^2$. $P(X_\infty\le0.3)=1-0.7^2=0.51$. $E=\dfrac{a}{a+b}=\dfrac13$, $\mathrm{Var}=\dfrac{ab}{(a+b)^2(a+b+1)}=\dfrac{2}{9\cdot4}=\dfrac1{18}=0.055556$.

(b) 정리 4: $\mathrm{Var}(W)=\dfrac{\sigma^2}{m(m-1)}=\dfrac{0.61}{1.3\cdot0.3}=\dfrac{0.61}{0.39}=1.564103$; $P(W=0)=q=0.4$.

(c) $M_n>0\iff$ 모든 $Y_i=2$이므로 $P(M_n>0)=2^{-n}$. $E[M_n]=\prod E[Y_i]=1^n=1$ (평균 1 곱 마팅게일, 05a Z7). $\sum_n P(M_n>0)=\sum2^{-n}<\infty$이므로 보렐-칸텔리로 $M_n>0$은 유한 번만 일어난다 ⇒ $M_n\to0$ a.s. (더 직접적으로, 첫 0이 나오면 영원히 0.) 비음 마팅게일이라 정리 1로 수렴은 보장되지만 극한 $0$의 기대 $0\ne E[M_n]=1$: 정리 1의 결론이 "a.s. 수렴"이지 "$L^1$ 수렴"이 아님을 보여 주는 가장 작은 예다. $M_n$이 UI가 아닌 이유: $E[M_n1_{\{M_n>K\}}]=1$ ($2^n>K$인 모든 $n$).

```python
from scipy.stats import beta
print(beta(1, 2).cdf(0.3), beta(1, 2).mean(), beta(1, 2).var())   # 0.51 0.3333 0.05556
print(0.61 / (1.3 * 0.3), extinction_probability([0.2, 0.3, 0.5]))  # 1.564103 0.4
```

</details>

### E2 유도 후 시뮬레이션 검증

베타-이항 공식 $P(R_n=k)=\dbinom nk\dfrac{a^{(k)}b^{(n-k)}}{(a+b)^{(n)}}$을 교환가능성으로 유도하고(정리 3 (ii)의 논법을 자기 말로), $(a,b)=(2,1)$과 $(1,1)$에서 $P(R_{10}=3)$을 손으로 계산한 뒤 항아리 $10^5$개로 검증하라 (`assert_close`, $k=4$).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

유도: 정리 3 (ii). 어떤 순서든 빨강 $k$개·파랑 $n-k$개인 열의 확률은 분모 $(a+b)^{(n)}$, 분자 $a^{(k)}b^{(n-k)}$로 같으므로(교환가능성) 그런 열의 개수 $\binom nk$를 곱한다.

$(2,1)$: $\dbinom{10}{3}\dfrac{2^{(3)}\,1^{(7)}}{3^{(10)}}=120\cdot\dfrac{(2\cdot3\cdot4)(1\cdot2\cdots7)}{3\cdot4\cdots12}=120\cdot\dfrac{24\cdot5040}{239500800}=\dfrac{2}{33}=0.060606$.

$(1,1)$: $R_{10}$은 $\{0,\dots,10\}$ 위 균등이므로 $P(R_{10}=3)=\dfrac1{11}=0.090909$. (공식으로도 $\binom{10}{3}\frac{3!\,7!}{11!/1}=\frac{10!}{11!}\cdot1=1/11$.)

```python
for a, b in [(2, 1), (1, 1)]:
    Xe = polya_urn(a, b, 10, rng, 100_000)
    R10 = np.rint(Xe[:, 10] * (a + b + 10)) - a          # X_10 = (a+R_10)/(a+b+10) 을 되돌린다
    est = mc_proportion(R10 == 3)
    print((a, b), est, betabinom(10, a, b).pmf(3))       # (2,1): ~0.0606 ± 0.0008,  (1,1): ~0.0909 ± 0.0009
    assert_close(est, betabinom(10, a, b).pmf(3), k=4, name=f"P(R_10=3), urn {(a, b)}")
```

</details>

### E3 가정을 깨보기

'뽑은 색의 **반대** 색을 추가'하는 항아리(Friedman형)로 규칙을 바꾸면 무엇이 달라지는가? $E[X_{n+1}\mid\mathcal F_n]$을 계산해 마팅게일인지 판정하고, $(1,1)$에서 $X_{1000}$의 히스토그램을 Beta$(1,1)$ 밀도와 비교하라. 극한은 무엇인가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

빨강 $R$개, 전체 $N=a+b+n$개일 때 빨강을 뽑을 확률은 $R/N$이고 그러면 파랑이 추가되어 빨강 수는 그대로, 파랑을 뽑으면(확률 $1-R/N$) 빨강이 하나 늘어난다. 따라서 $E[R_{n+1}\mid\mathcal F_n]=R+(1-R/N)$이고

$$E[X_{n+1}\mid\mathcal F_n]=\frac{R+1-R/N}{N+1}=\frac{X_nN+1-X_n}{N+1}=X_n+\frac{1-2X_n}{N+1}.$$

$X_n>1/2$이면 아래로, $X_n<1/2$이면 위로 끌리는 **평균회귀**이지 마팅게일이 아니다. '같은 색 추가'가 만들던 교환가능성이 사라지므로 베타-이항 구조도 없다. 극한은 확률변수가 아니라 상수 $1/2$ (a.s.): 밀어내는 힘 $(1-2X_n)/(N+1)$은 $1/n$ 규모로 작아지지만 합이 발산하므로 계속 작동한다. 정확 DP(분포 $R_n$을 시간축으로 전파)로 $(1,1)$에서 $\mathrm{sd}(X_{1000})=0.00912$ — Beta$(1,1)$의 $0.2887$과 비교하면 히스토그램이 $1/2$ 근처에 뾰족하게 몰린다 ($\sqrt n\cdot\mathrm{sd}\approx0.288\approx1/\sqrt{12}$: 실제로 $\sqrt n(X_n-1/2)$는 CLT 규모로 정규분포에 접근한다).

```python
def opposite_color_urn(a, b, n_draws, rng, n_paths):
    """뽑은 색의 반대 색을 추가하는 항아리 (Friedman형). 빨간 비율, shape (n_paths, n_draws+1)."""
    red, total = np.full(n_paths, a, dtype=float), float(a + b)
    X = np.empty((n_paths, n_draws + 1)); X[:, 0] = red / total
    for k in range(n_draws):
        red += rng.random(n_paths) >= red / total    # 파랑을 뽑으면 빨강 +1
        total += 1.0
        X[:, k + 1] = red / total
    return X

Xo = opposite_color_urn(1, 1, N_DRAWS, rng, N_PATHS)
print(Xo[:, -1].mean(), Xo[:, -1].std(), 1 / np.sqrt(12))     # ~0.5000  ~0.0091  0.2887
fig, ax = plt.subplots()
plot_hist_vs_pdf(Xo[:, -1], xs, beta(1, 1).pdf(xs), ax=ax, bins=40, label_pdf="Beta(1,1) pdf (same-colour rule)")
ax.set_title("E3. Opposite-colour urn (1,1): X_1000 collapses to 1/2"); ax.set_xlabel("X_1000"); ax.legend()
```

</details>

## 7. 정리

1. **$L^1$ 유계 마팅게일은 a.s. 수렴한다** (정리 1). 증명의 핵심은 상향 교차 부등식 $(b-a)E[U_n[a,b]]\le E[(M_n-a)^-]$: "싸게 사서 비싸게 파는" 전략이 공정 게임에서 돈을 벌 수 없으므로 어떤 구간도 무한히 건널 수 없다. 비음 마팅게일은 항상 수렴; SRW는 $L^1$ 무계($E|S_n|\sim\sqrt{2n/\pi}$)라 수렴하지 않는다.
2. **$L^2$ 유계이면 $L^2$ 수렴까지** 되어 $E[M_\infty]=M_0$, $\mathrm{Var}(M_\infty)=\sum E[\Delta_k^2]$ (직교 증분·피타고라스). 일반적으로 기대값 보존에는 UI가 필요하다.
3. **폴리아 항아리**: 비율은 Beta$(a,b)$ 확률변수로 수렴하고 유한 $n$의 분포는 베타-이항이다(교환가능성). 초기 뽑기가 운명을 정한다 — 그림 1의 경로 15개.
4. **골턴-왓슨** $Z_n/m^n\to W$: $E[W]=1$, $\mathrm{Var}(W)=\sigma^2/(m(m-1))$, $P(W=0)=q$ (Kesten–Stigum 조건 $E[X\log X]<\infty$ 아래). $(0.2,0.3,0.5)$: $1.564$, $0.4$.
5. **우도비** $L_n\to0$ a.s.이지만 $E[L_n]=1$: UI 결여의 표준 예이며, $n=200$에서 MC 평균이 $1$ 아래로 무너지는 이유다(분산 $e^{50}$). 불편추정량도 분산이 폭발하면 무의미하다.

**Trap 카드**
- Q: 폴리아 항아리에서 $n\to\infty$이면 빨간 공 비율은 $a/(a+b)$로 수렴하는가?
- A: 아니다. 비율은 수렴하지만 극한은 Beta$(a,b)$ 확률변수이고 **평균만** $a/(a+b)$이다. 같은 항아리를 다시 돌리면 다른 값에 정착한다 — 그림 1의 15개 경로가 그 증거.

**다음 노트북: 05e** — 수렴하는 마팅게일이 "얼마나 벗어날 수 있는가"를 정량화한다: 둡 최대 부등식(경로 최댓값의 꼬리)과 아주마 부등식(유계 차분 마팅게일의 집중). 선택 노트북이므로 건너뛰면 05r 복습으로 간다.

log/progress.md 한 줄 템플릿:
`- [ ] 05d 마팅게일 수렴과 폴리아 항아리 — YYYY-MM-DD, 예측 6개 중 __개 적중, 막힌 곳: ____, 다시 볼 것: 상향 교차 부등식의 베팅 해석 / a.s. 수렴 vs UI(우도비)`